In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, col, row_number, count
from pyspark.sql.window import Window
import re

# --------------------------------------------------
# Configuration
# --------------------------------------------------
raw_csv_path = "/Volumes/bronze_dev/global_mart_retail/raw_data/Sample - Superstore.csv"
table_name = "bronze_dev.global_mart_retail.raw_data"

business_keys = ["order_id", "product_id"]

# --------------------------------------------------
# Read CSV
# --------------------------------------------------
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .csv(raw_csv_path)
)


# --------------------------------------------------
# Clean column names (Unity Catalog safe)
# --------------------------------------------------
def clean_column(col_name):
    col_name = col_name.lower()
    col_name = re.sub(r"[ -,;{}()\n\t=]", "_", col_name)
    return col_name.strip("_")

df = df.toDF(*[clean_column(c) for c in df.columns])

# --------------------------------------------------
# Add Bronze metadata
# --------------------------------------------------
df = (
    df.withColumn("ingestion_ts", current_timestamp())
      .withColumn("source_file_path", col("_metadata.file_path"))
)

# --------------------------------------------------
# Deterministic Deduplication
# Keep greatest row_id per (order_id, product_id)
# --------------------------------------------------
window_spec = (
    Window
    .partitionBy(*business_keys)
    .orderBy(col("row_id").desc())
)


df_dedup = (
    df.withColumn("rn", row_number().over(window_spec))
      .filter(col("rn") == 1)
      .drop("rn")
)

display(df_dedup.limit(5))


In [0]:
from pyspark.sql.functions import count

df_dedup.groupBy("order_id", "product_id", "ingestion_ts") \
        .agg(count("*").alias("dup")) \
        .filter("dup > 1") \
        .show()

In [0]:
# --------------------------------------------------
# Build MERGE condition for composite key
# --------------------------------------------------
merge_condition = " AND ".join(
    [f"tgt.{k} = src.{k}" for k in business_keys]
)


# --------------------------------------------------
# Incremental Load (MERGE)
# --------------------------------------------------
if spark.catalog.tableExists(table_name):
    delta_table = DeltaTable.forName(spark, table_name)

    (
        delta_table.alias("tgt")
        .merge(
            df_dedup.alias("src"),
            merge_condition
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        df_dedup.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

# --------------------------------------------------
# Validation
# --------------------------------------------------
spark.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id, product_id) AS distinct_business_keys,
        ingestion_ts
    FROM {table_name}
    group by ingestion_ts
""").show()


In [0]:
from pyspark.sql.functions import count

df_dedup.groupBy("order_id", "product_id") \
        .agg(count("*").alias("dup")) \
        .filter("dup > 1") \
        .show()


In [0]:
spark.sql(f"""
    SELECT 
    order_id, product_id,
        COUNT(*) AS dups
    FROM {table_name}
    group by order_id, product_id
    having COUNT(*) > 1
""").show()

In [0]:
%sql
select * from bronze_dev.global_mart_retail.raw_data limit 10;

In [0]:
%sql
DESCRIBE EXTENDED bronze_dev.global_mart_retail.raw_data

In [0]:
%sql
WITH duplicates AS (
  SELECT
    order_id,
    product_id,
    COUNT(*) AS duplicate_count
  FROM bronze_dev.global_mart_retail.raw_data
  GROUP BY order_id, product_id
  HAVING COUNT(*) > 1
)

SELECT
  b.*,
  concat_ws(' | ', b.order_id, b.product_id) AS concat_key
FROM bronze_dev.global_mart_retail.raw_data b 
INNER JOIN duplicates d
ON b.order_id = d.order_id
AND b.product_id = d.product_id
ORDER BY b.order_id, b.product_id DESC;

In [0]:
%sql
SELECT
  order_id,
  product_id,
  MAX(row_id) AS max_row_id
FROM bronze_dev.global_mart_retail.raw_data
GROUP BY order_id, product_id
